# Fine-tuning CodeLlama-7B-Instruct con QLoRA (T4 16GB)

Este notebook entrena un adapter LoRA (QLoRA 4-bit) para un generador educativo de codigo Python.

## Runtime requerido
- En Colab: `Runtime > Change runtime type > GPU`
- GPU esperada: **T4 (16GB)**


## 1) Setup GPU / info

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

## 2) Install dependencies

In [ ]:
# Instalacion robusta para Colab T4 (evita conflictos torch/bnb/triton)
!pip -q uninstall -y torch torchvision torchaudio transformers peft trl accelerate bitsandbytes triton
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1
!pip -q install transformers==4.44.2 peft==0.12.0 trl==0.10.1 accelerate==0.33.0 datasets==2.21.0 bitsandbytes==0.43.3 triton==2.3.1 sentencepiece==0.1.99 protobuf==4.25.3

print('Instalacion completada. IMPORTANTE: Runtime > Restart runtime antes de seguir.')

## 3) Login HF (opcional)
Solo necesario si el modelo/dataset requiere auth o si quieres pushear a Hub.

In [ ]:
from huggingface_hub import notebook_login
# notebook_login()  # Descomenta si necesitas login

## 3.5) Sanity check de entorno (obligatorio)
Ejecuta esta celda **despues de reiniciar runtime**. Si falla, no sigas al bloque de modelo.

In [ ]:
import torch
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('cuda_available:', torch.cuda.is_available())

import bitsandbytes as bnb
print('bnb:', bnb.__version__)

import triton
print('triton:', triton.__version__)

# Diagnostico bnb
!python -m bitsandbytes

## 4) Load dataset (Drive o subida manual)

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

USE_DRIVE = True  # True: leer desde Drive | False: subir manual

DRIVE_TRAIN_PATH = '/content/drive/MyDrive/finetuning/train.jsonl'
DRIVE_VAL_PATH = '/content/drive/MyDrive/finetuning/val.jsonl'

LOCAL_DATA_DIR = Path('/content/data/finetuning')
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_dst = LOCAL_DATA_DIR / 'train.jsonl'
val_dst = LOCAL_DATA_DIR / 'val.jsonl'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy(DRIVE_TRAIN_PATH, train_dst)
    shutil.copy(DRIVE_VAL_PATH, val_dst)
else:
    print('Sube train.jsonl')
    up_train = files.upload()
    train_name = list(up_train.keys())[0]
    shutil.move(train_name, train_dst)

    print('Sube val.jsonl')
    up_val = files.upload()
    val_name = list(up_val.keys())[0]
    shutil.move(val_name, val_dst)

print('Train:', train_dst, 'exists=', train_dst.exists())
print('Val:', val_dst, 'exists=', val_dst.exists())

## 5) Preprocess: prompt instruct para CodeLlama

`MAX_SEQ_LENGTH=1024` por defecto (T4-friendly). Si OOM, baja a 768 o 512.

In [ ]:
from datasets import load_dataset

data_files = {
    'train': str(train_dst),
    'validation': str(val_dst),
}
raw_ds = load_dataset('json', data_files=data_files)
raw_ds

In [ ]:
MODEL_NAME = 'codellama/CodeLlama-7b-Instruct-hf'
MAX_SEQ_LENGTH = 1024

SYSTEM_TEXT = (
    'Eres un asistente educativo de Python para ciencia de datos. '
    'Debes responder con formato estructurado y codigo ejecutable.'
)

def format_example(example):
    instruction = str(example.get('instruction', '')).strip()
    inp = str(example.get('input', '')).strip()
    out = str(example.get('output', '')).strip()
    prompt = (
        f"<s>[INST] <<SYS>>\n{SYSTEM_TEXT}\n<</SYS>>\n\n"
        f"Instruction: {instruction}\n"
        f"Input: {inp}\n\n"
        f"Genera la respuesta completa con secciones OBJETIVO, CODIGO y EXPLICACION. [/INST]\n"
        f"{out}</s>"
    )
    return {'text': prompt}

ds = raw_ds.map(format_example, remove_columns=raw_ds['train'].column_names)
ds

## 6) Load model (4-bit) + tokenizer

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError('CUDA no disponible. En Colab selecciona Runtime > Change runtime type > GPU y reinicia.')

gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
print('Model + tokenizer loaded')

## 7) Config LoRA

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
lora_config

## 8) Train (SFTTrainer)

Config recomendada T4: batch pequeno + grad accumulation, fp16, gradient_checkpointing y paged_adamw_8bit.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

output_dir = '/content/models/codellama-edugen'

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,  # opcion: 3
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,  # batch efectivo 16
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    evaluation_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    fp16=True,
    bf16=False,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    lr_scheduler_type='cosine',
    seed=42,
    report_to='none',
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    peft_config=lora_config,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
)

train_result = trainer.train()
train_result

## 9) Evaluate (eval loss)

In [ ]:
metrics = trainer.evaluate()
print(metrics)

## 10) Save artifacts (adapter + tokenizer + args + logs)

In [ ]:
import json
from pathlib import Path

save_dir = Path('/content/models/codellama-edugen')
save_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

with (save_dir / 'training_args.json').open('w') as f:
    json.dump(training_args.to_dict(), f, indent=2)

if trainer.state is not None:
    with (save_dir / 'trainer_state.json').open('w') as f:
        json.dump(trainer.state.log_history, f, indent=2)

print('Saved to', save_dir)
print('Files:', [p.name for p in save_dir.iterdir()])

## 11) Quick generation test (2-3 prompts)

In [ ]:
import torch

def make_test_prompt(topic, level, context, kind):
    return (
        '<s>[INST] <<SYS>>\nEres un asistente educativo de Python para ciencia de datos.\n<</SYS>>\n\n'
        'Instruction: Genera un ejercicio educativo de Python con secciones OBJETIVO, CODIGO y EXPLICACION.\n'
        f'Input: Tema: {topic}, Nivel: {level}, Contexto: {context}, Tipo: {kind}\n\n'
        'Responde en espanol y con codigo ejecutable. [/INST]'
    )

tests = [
    ('pandas_groupby', 'principiante', 'deportes', 'tutorial'),
    ('matplotlib_basico', 'intermedio', 'ciencia', 'desafio'),
    ('pandas_filtrado', 'avanzado', 'finanzas', 'mini-proyecto'),
]

model.eval()
for t in tests:
    prompt = make_test_prompt(*t)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=320,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print('--- TEST:', t, '---')
    print(text[:2000])
    print('Has OBJETIVO:', 'OBJETIVO' in text)
    print('Has CODIGO:', 'CODIGO' in text)
    print('Has EXPLICACION:', 'EXPLICACION' in text)
    print()

## 12) Export: zip del modelo/adapters para descargar

In [ ]:
import shutil
from google.colab import files

zip_base = '/content/models/codellama-edugen'
zip_file = shutil.make_archive(zip_base, 'zip', '/content/models/codellama-edugen')
print('Zip created:', zip_file)
files.download(zip_file)

## Troubleshooting OOM (T4)
- Baja `MAX_SEQ_LENGTH` de 1024 a 768 o 512.
- Reduce `per_device_train_batch_size` a 1.
- Sube `gradient_accumulation_steps` para mantener batch efectivo.
- Mant?n `gradient_checkpointing=True`.
- Limpia memoria entre pruebas: `gc.collect(); torch.cuda.empty_cache()`.
